# 📘 Notebook 16: Linked Lists

### Course: *From Zero to Data Structures & Algorithms in Python*
**Instructor:** Ioannis Tsioulis

*Module D: Data Structures · Notebook 2 of 4 in this module · (16 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain how a **linked list** stores data as a chain of **nodes**
- Contrast it with Python's built-in list, which is a dynamic array
- Build and traverse a linked list by hand, with nodes represented as dictionaries
- Implement prepend, append and delete, and state the cost of each
- Write a `Node` class and a full `LinkedList` class
- Decide when a linked list is the better choice, and when it is not

> **Prerequisites:** Notebooks 07 (classes), 11 (Big O) and 15 (stacks and queues).
>
> 🔭 **Why this notebook matters:** the linked list introduces one of the great ideas of computer science, data that is held together by **references** rather than by position. The trees of Notebook 18 are built on exactly the same idea.

## 1. Definition and motivation

### The problem with arrays
Python's built-in `list` is a **dynamic array**: its elements sit side by side in one continuous block of memory. That is why reading `numbers[i]` is O(1). It is also why inserting at the front is O(n): to make room at position 0, **every** other element has to move one place to the right. We measured this in Notebook 11.

### Intuition first
Picture a treasure hunt. Each clue contains a prize and **the location of the next clue**. The clues can be scattered anywhere. All you need is the location of the first one, and from there each clue leads you to the next.

To add a new clue at the start of the hunt you do not move any of the others. You write one new clue that points to the old first clue.

### Formal definition
A **singly linked list** is a linear data structure in which each element is stored in a separate object called a **node**. Each node contains two fields:

- **data**: the actual value;
- **next**: a reference to the next node in the sequence.

The last node has its `next` set to `None`, marking the end of the list. A variable called the **head** refers to the first node. It is our only way in.

```
head
 |
 v
[10 | next] -> [20 | next] -> [30 | next] -> [40 | None]
```

### Python's list versus a singly linked list

| Operation / property | Python `list` | Singly linked list |
|---|---|---|
| Underlying implementation | dynamic array | chain of nodes |
| Memory layout | continuous | scattered |
| Insert or delete at the beginning | O(n) | **O(1)** |
| Insert or delete at the end | O(1) | O(1) with a tail reference, O(n) without |
| Access by index (`x[i]`) | **O(1)** | O(n) |
| Memory overhead | low | one reference per element |

Neither is better. They make opposite trade-offs.

## 2. Representing a node without a class

To understand how references work, we begin with the simplest possible representation: a node is a dictionary with two keys.

```python
node = {"data": value, "next": reference_to_next_node_or_None}
```

In [ ]:
# Create four individual nodes. They are not connected yet.
n1 = {"data": 10, "next": None}
n2 = {"data": 20, "next": None}
n3 = {"data": 30, "next": None}
n4 = {"data": 40, "next": None}

# Link them together
n1["next"] = n2
n2["next"] = n3
n3["next"] = n4
# n4["next"] remains None: it is the last node

head = n1          # head always refers to the first node

The whole list is now reachable from `head`. Following the references by hand:

In [ ]:
print(head["data"])                              # 10
print(head["next"]["data"])                      # 20
print(head["next"]["next"]["data"])              # 30
print(head["next"]["next"]["next"]["data"])      # 40
print(head["next"]["next"]["next"]["next"])      # None: the end

### Traversal
Nobody wants to write `["next"]["next"]["next"]`. To visit every node we use a variable, `current`, that starts at the head and **hops** from node to node until it falls off the end.

This loop is the heart of every linked-list operation. Study it carefully.

In [ ]:
def print_linked_list(head):
    current = head
    while current is not None:
        print(current["data"], end=" -> ")
        current = current["next"]        # hop to the next node
    print("None")

print_linked_list(head)

> ⚠️ **Common pitfalls**
>
> - Forgetting `current = current["next"]`. The loop then stays on the same node forever.
> - Moving `head` itself during a traversal. If you lose the reference to the first node, the list is gone. Always walk with a separate variable.

## 3. Core operations

### 3.1 Prepend: O(1)
To insert at the **front**, create a node whose `next` is the current head. The new node becomes the head. No existing node is touched, and the cost does not depend on the length of the list.

```
before:            head -> [10] -> [20] -> ...
after:   head -> [5] -> [10] -> [20] -> ...
```

In [ ]:
def prepend(head, value):
    new_node = {"data": value, "next": head}
    return new_node                     # the new node is the new head

head = prepend(head, 5)
print_linked_list(head)

### 3.2 Append: O(n)
To insert at the **end** we must first *find* the end, and the only way is to walk there from the head. That walk makes append O(n).

The empty list needs separate treatment: if there is no node at all, the new node is the head.

In [ ]:
def append(head, value):
    new_node = {"data": value, "next": None}

    if head is None:                    # empty list
        return new_node

    current = head
    while current["next"] is not None:  # stop ON the last node, not after it
        current = current["next"]
    current["next"] = new_node
    return head                         # the head is unchanged

head = append(head, 50)
print_linked_list(head)

### 3.3 Delete the first occurrence: O(n)
To remove a node we make the node **before** it skip over it.

```
before:  [10] -> [20] -> [30]
after:   [10] ---------> [30]        the node 20 is no longer reachable
```
That is why the loop looks one step ahead, at `current["next"]`: we need to be standing on the *previous* node when we find the one to remove. Deleting the head is a special case, since it has no previous node.

In [ ]:
def delete(head, value):
    if head is None:
        return None

    if head["data"] == value:           # deleting the head node
        return head["next"]

    current = head
    while current["next"] is not None:
        if current["next"]["data"] == value:
            current["next"] = current["next"]["next"]     # skip over it
            return head
        current = current["next"]

    return head                         # value not found: nothing changes

head = delete(head, 20)
print_linked_list(head)

> 🧠 Each function **returns the head**, because the operation may change which node is first. The caller must write `head = prepend(head, 5)`. This is slightly awkward, and it is one of the things a class will tidy up.

## 4. A proper implementation with classes

The dictionary version is excellent for seeing how references work. For real use we want what Notebook 15 taught us: a class with a clear interface that hides the machinery.

We need two classes: a `Node`, and a `LinkedList` that owns the head.

In [ ]:
class Node:
    def __init__(self, data):
        self.data = data
        self.next = None

class LinkedList:
    def __init__(self):
        self.head = None
        self.size = 0

    def prepend(self, value):
        new_node = Node(value)
        new_node.next = self.head
        self.head = new_node
        self.size += 1

    def append(self, value):
        new_node = Node(value)
        self.size += 1
        if self.head is None:
            self.head = new_node
            return
        current = self.head
        while current.next is not None:
            current = current.next
        current.next = new_node

    def find(self, value):
        current = self.head
        while current is not None:
            if current.data == value:
                return True
            current = current.next
        return False

    def delete(self, value):
        if self.head is None:
            return False
        if self.head.data == value:
            self.head = self.head.next
            self.size -= 1
            return True
        current = self.head
        while current.next is not None:
            if current.next.data == value:
                current.next = current.next.next
                self.size -= 1
                return True
            current = current.next
        return False

    def __str__(self):
        parts = []
        current = self.head
        while current is not None:
            parts.append(str(current.data))
            current = current.next
        return " -> ".join(parts + ["None"])

In [ ]:
numbers = LinkedList()
numbers.append(10)
numbers.append(20)
numbers.append(30)
numbers.prepend(5)
print(numbers)

print("Contains 20?", numbers.find(20))
print("Contains 99?", numbers.find(99))

numbers.delete(20)
print(numbers)
print("Size:", numbers.size)

Compare the two versions. `current["next"]` became `current.next`, and the head now lives inside the object, so the caller simply writes `numbers.prepend(5)`. The logic is identical.

Keeping a `size` attribute up to date gives us the length in **O(1)**, instead of walking the whole chain to count it.

## 5. Measuring the trade-off

The table in section 1 made two claims. Let us test both.

### Claim 1: inserting at the front is O(1) for a linked list, O(n) for a Python list

In [ ]:
import time

n = 50_000

start = time.perf_counter()
python_list = []
for i in range(n):
    python_list.insert(0, i)
list_time = time.perf_counter() - start

start = time.perf_counter()
linked = LinkedList()
for i in range(n):
    linked.prepend(i)
linked_time = time.perf_counter() - start

print(f"Python list, insert(0, x): {list_time:.4f} seconds")
print(f"Linked list, prepend(x):   {linked_time:.4f} seconds")

### Claim 2: access by position is O(1) for a Python list, O(n) for a linked list
A linked list has no index. To reach element number `i` we must hop `i` times from the head.

In [ ]:
def get_at(linked_list, index):
    current = linked_list.head
    for i in range(index):
        current = current.next
    return current.data

positions = [n // 2] * 200           # read the middle element 200 times

start = time.perf_counter()
for p in positions:
    python_list[p]
list_time = time.perf_counter() - start

start = time.perf_counter()
for p in positions:
    get_at(linked, p)
linked_time = time.perf_counter() - start

print(f"Python list, x[i]:        {list_time:.6f} seconds")
print(f"Linked list, walk to i:   {linked_time:.4f} seconds")

### Observed result
Each structure wins one contest decisively and loses the other. This is the essential lesson of data structures: **there is no best structure, only the best structure for the operations you need most**.

| Choose a linked list when... | Choose a Python list when... |
|---|---|
| you insert and remove mostly at the front | you need access by index |
| the size changes constantly | you mostly add at the end |
| you are building a stack or a queue | you need to sort or to binary search |

> 🧠 Honest advice for everyday Python: the built-in list and `deque` are so well optimised that you will rarely need to write a linked list yourself. You study it for the idea of **nodes joined by references**, which is the foundation of trees, graphs and much of what lies beyond this course.

---
## ✏️ Exercises

Unless stated otherwise, use the `Node` and `LinkedList` classes of section 4.

### Exercise 1 (Build and print)
Create a linked list containing the names `"Anna"`, `"Kostas"` and `"Eleni"`, in that order, then add `"Zoe"` at the front. Print it.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
names = LinkedList()
names.append("Anna")
names.append("Kostas")
names.append("Eleni")
names.prepend("Zoe")
print(names)
```
</details>

### Exercise 2 (Count the nodes)
Write a function `count_nodes(linked_list)` that counts the nodes by traversing the list, **without** using the `size` attribute. Check that it agrees with `size`.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def count_nodes(linked_list):
    count = 0
    current = linked_list.head
    while current is not None:
        count += 1
        current = current.next
    return count

numbers = LinkedList()
for value in [4, 8, 15, 16, 23, 42]:
    numbers.append(value)

print(count_nodes(numbers), numbers.size)
```
</details>

### Exercise 3 (Sum and maximum)
Write two functions, `sum_values(linked_list)` and `max_value(linked_list)`, for a linked list of numbers.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def sum_values(linked_list):
    total = 0
    current = linked_list.head
    while current is not None:
        total += current.data
        current = current.next
    return total

def max_value(linked_list):
    current = linked_list.head
    largest = current.data
    while current is not None:
        if current.data > largest:
            largest = current.data
        current = current.next
    return largest

print(sum_values(numbers))
print(max_value(numbers))
```

*Every one of these functions has the same skeleton: start at the head, loop while `current is not None`, do something, hop to the next node.*
</details>

### Exercise 4 (Convert to a Python list)
Write a function `to_list(linked_list)` that returns an ordinary Python list with the same values in the same order.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def to_list(linked_list):
    result = []
    current = linked_list.head
    while current is not None:
        result.append(current.data)
        current = current.next
    return result

print(to_list(numbers))
```
</details>

### Exercise 5 (Remove the first node)
Write a function `remove_first(linked_list)` that removes the head node and **returns its value**, or `None` if the list is empty. What is its Big O?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def remove_first(linked_list):
    if linked_list.head is None:
        return None
    value = linked_list.head.data
    linked_list.head = linked_list.head.next
    linked_list.size -= 1
    return value

print(remove_first(numbers))
print(numbers)
```

*O(1): no traversal is needed. Together with `prepend`, this gives a complete **stack** in which both operations are O(1).*
</details>

### Exercise 6 (Insert after a value)
Write a function `insert_after(linked_list, target, value)` that inserts a new node containing `value` immediately after the first node containing `target`. If `target` is not found, do nothing and return `False`.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def insert_after(linked_list, target, value):
    current = linked_list.head
    while current is not None:
        if current.data == target:
            new_node = Node(value)
            new_node.next = current.next     # 1. the new node points onwards
            current.next = new_node          # 2. the current node points to the new one
            linked_list.size += 1
            return True
        current = current.next
    return False

insert_after(numbers, 15, 99)
print(numbers)
```

*The order of the two assignments matters. If you set `current.next = new_node` first, the reference to the rest of the list is lost.*
</details>

### Exercise 7 (A tail reference)
`append` is O(n) because it has to walk to the end. Write a class `LinkedListWithTail` that keeps a second attribute, `tail`, referring to the last node, so that `append` becomes O(1). Implement `append`, `prepend` and `__str__`. Be careful with the empty list, where the new node is both the head and the tail.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
class LinkedListWithTail:
    def __init__(self):
        self.head = None
        self.tail = None

    def append(self, value):
        new_node = Node(value)
        if self.head is None:
            self.head = new_node
            self.tail = new_node
        else:
            self.tail.next = new_node
            self.tail = new_node

    def prepend(self, value):
        new_node = Node(value)
        new_node.next = self.head
        self.head = new_node
        if self.tail is None:
            self.tail = new_node

    def __str__(self):
        parts = []
        current = self.head
        while current is not None:
            parts.append(str(current.data))
            current = current.next
        return " -> ".join(parts + ["None"])

items = LinkedListWithTail()
items.append(1)
items.append(2)
items.prepend(0)
items.append(3)
print(items)
```

*With O(1) `append` at the tail and O(1) removal at the head, this structure is an efficient **queue**.*
</details>

### Exercise 8 (Reverse a linked list, a little harder)
Write a function `reverse(linked_list)` that reverses the list **in place**, by redirecting the `next` references, without creating any new node. (Hint: walk through the list with three variables, `previous`, `current` and `following`. At each node, make it point backwards.)

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def reverse(linked_list):
    previous = None
    current = linked_list.head
    while current is not None:
        following = current.next      # remember where to go next
        current.next = previous       # turn this node around
        previous = current            # step forward
        current = following
    linked_list.head = previous       # the old last node is the new head

letters = LinkedList()
for letter in "ABCDE":
    letters.append(letter)

print(letters)
reverse(letters)
print(letters)
```

*This is one of the most famous interview questions. Draw the nodes and the three variables on paper and move them step by step. It is O(n), with no extra memory.*
</details>

## 🔑 Key takeaways

- A **linked list** is a chain of **nodes**. Each node holds `data` and a reference, `next`, to the following node. The last one points to `None`.
- The **head** is the only entry point. Never lose it.
- **Traversal** is the basic pattern: `current = head`, loop while `current is not None`, then `current = current.next`.
- **Prepend is O(1).** Append and delete need a walk, so they are O(n), unless a **tail** reference is kept.
- There is **no index**: reaching position `i` costs O(n).
- Python's built-in `list` is a dynamic array, **not** a linked list. The two make opposite trade-offs.
- There is no best data structure, only the best one for the operations you need.

---
**Next:** *Notebook 17: Hash Tables*, where we finally explain how a dictionary finds a value instantly.

---
*© Ioannis Tsioulis. *From Zero to Data Structures & Algorithms in Python*. Prepared for educational use.*